In [1]:
# Importing the Libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
import nltk
import re
from nltk.corpus import stopwords
import string

In [2]:
# Loading the Dataset
data = pd.read_csv('consumercomplaints.csv')

In [3]:
# Analyse the Top 5 rows of the Data
data.head()

,Unnamed: 0,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative
0,0,2022-11-11,Mortgage,Conventional home mortgage,Trouble during payment process,NaN,NaN
1,1,2022-11-23,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,Reporting company used your report improperly,NaN
2,2,2022-11-16,Mortgage,VA mortgage,Trouble during payment process,NaN,NaN
3,3,2022-11-15,Checking or savings account,Checking account,Managing an account,Fee problem,"Hi, I have been banking with Wells Fargo for o..."
4,4,2022-11-07,Mortgage,Other type of mortgage,Trouble during payment process,NaN,NaN


In [4]:
# The dataset contains an Unnamed column. Let's remove the column and move further
data = data.drop(['Unnamed: 0'], axis = 1)

In [5]:
# Now let’s have a look if the dataset contains null values
data.isnull().sum()

Date received                         0
Product                               0
Sub-product                      235294
Issue                                 0
Sub-issue                        683355
Consumer complaint narrative    1987977
dtype: int64

In [6]:
# The dataset contains so many null values. Let's drop all the rows containing null values and move further
data.dropna(inplace= True)

##### The product column in the dataset contains the labels. Here the labels represent the nature of the complaints reported by the consumers.

In [7]:
# Let’s have a look at all the labels and their frequency
data['Product'].value_counts()

Product
Credit reporting, credit repair services, or other personal consumer reports    507582
Debt collection                                                                 192045
Credit card or prepaid card                                                      80410
Checking or savings account                                                      54192
Student loan                                                                     32697
Vehicle loan or lease                                                            19874
Payday loan, title loan, or personal loan                                         1008
Name: count, dtype: int64

#### Training Consumer Complaint Classification Model

In [8]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


nltk.download('stopwords')

stopword = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean(text):
    text = str(text).lower()  # Convert to lowercase
    text = re.sub(r'\[.*?\]', '', text)  # Remove text inside brackets
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>+', '', text)  # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)  # Remove punctuation
    text = re.sub(r'\n', '', text)  # Remove newline characters
    text = re.sub(r'\w*\d\w*', '', text)  # Remove words with numbers

    # Tokenization and Stopword Removal
    text = text.split()
    text = [word for word in text if word not in stopword]

    # Stemming
    text = [stemmer.stem(word) for word in text]

    return " ".join(text)  # Convert list back to string

# Apply cleaning function
data["Consumer complaint narrative"] = data["Consumer complaint narrative"].apply(clean)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Kamran\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
# Now, let’s split the data into training and test sets
data = data[['Consumer complaint narrative', 'Product']]
x = np.array(data['Consumer complaint narrative'])
y = np.array(data['Product'])

In [11]:
cv = CountVectorizer()
X = cv.fit_transform(x)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 42)

In [12]:
# Now let’s train the Machine Learning model using the Stochastic Gradient Descent classification algorithm
sgdmodel = SGDClassifier()
sgdmodel.fit(X_train, y_train)

SGDClassifier()

In [13]:
# Now let’s use our trained model to make predictions
user = input("Enter a Text: ")
data = cv.transform([user]).toarray()
output = sgdmodel.predict(data)
print(output)

Enter a Text:  This scam relies on urgency and fear, pressuring users to pay for goods or services they have not ordered. Finance departments are common targets for this type of attack.


['Credit reporting, credit repair services, or other personal consumer reports']


In [14]:
user = input("Enter a Text: ")
data = cv.transform([user]).toarray()
output = sgdmodel.predict(data)
print(output)

Enter a Text:  This scam uses various messages to convince users to provide personal details. It might claim that the user is in the wrong Council Tax band and is owed back payments, or it might ask for bank details to provide a refund.


['Credit reporting, credit repair services, or other personal consumer reports']


#### Summary
Consumer Complaint Classification is helpful for consumer care departments as they receive thousands of complaints daily, so classifying them helps identify complaints that need to be solved first to reduce the loss of the consumer.